In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from transformers import CLIPModel, CLIPProcessor
from torch.utils.data import DataLoader, Subset
from peft import get_peft_model, LoraConfig, TaskType
from torchattacks import PGD
from collections import defaultdict
import random

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
path = '/content/drive/MyDrive/DataSets2/'

In [4]:
import torch
import torchvision

In [5]:
##transformation

import torchvision.transforms as transforms

pre_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],std=[0.26862954, 0.26130258, 0.27577711])
])

In [6]:
##dataset
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

test_dataset=datasets.CIFAR10(root=path, train=False, download=True, transform=pre_transform)
test_loader=DataLoader(test_dataset, batch_size=64, shuffle=False)

In [7]:
##getting a mini train data
def balanced_data(dataset, target_size):
    num_classes=10
    per_class=target_size//num_classes
    class_indices=defaultdict(list)

    for idx, (_, label) in enumerate(dataset):
        class_indices[label].append(idx)

    selected_indices=[]
    for cls in range(num_classes):
        selected_indices.extend(random.sample(class_indices[cls], per_class))

    random.shuffle(selected_indices)
    return Subset(dataset, selected_indices)

In [8]:
test_dataset2=balanced_data(test_dataset,200)
test_loader2=DataLoader(test_dataset2, batch_size=64, shuffle=False)

In [9]:
import os
import torch

save_dir="/content/drive/MyDrive/clip_lora_adv_datset"

# Load tensors
adv_train_images=torch.load(os.path.join(save_dir, "train_images.pt"))
adv_train_labels=torch.load(os.path.join(save_dir, "train_labels.pt"))
adv_val_images=torch.load(os.path.join(save_dir, "val_images.pt"))
adv_val_labels=torch.load(os.path.join(save_dir, "val_labels.pt"))
adv_test_images=torch.load(os.path.join(save_dir, "test_images.pt"))
adv_test_labels=torch.load(os.path.join(save_dir, "test_labels.pt"))

adv_train_dataset=torch.utils.data.TensorDataset(adv_train_images, adv_train_labels)
adv_val_dataset=torch.utils.data.TensorDataset(adv_val_images, adv_val_labels)
adv_test_dataset=torch.utils.data.TensorDataset(adv_test_images, adv_test_labels)

# Create loaders
batch_size=64
adv_train_loader=torch.utils.data.DataLoader(adv_train_dataset, batch_size=batch_size, shuffle=True)
adv_val_loader=torch.utils.data.DataLoader(adv_val_dataset, batch_size=batch_size, shuffle=False)
adv_test_loader=torch.utils.data.DataLoader(adv_test_dataset, batch_size=batch_size, shuffle=False)

model SetUp

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

##clip model and its processor from hugging face
import torch
from transformers import CLIPProcessor, CLIPModel
clip_model_name="openai/clip-vit-base-patch32"
##Loading pretrained weights 
clip_model=CLIPModel.from_pretrained(clip_model_name)
##clip processor 
clip_processor=CLIPProcessor.from_pretrained(clip_model_name)

for param in clip_model.parameters():
    param.requires_grad = False

##LoRa
lora_config=LoraConfig(r=8,lora_alpha=32,target_modules=["q_proj", "v_proj"],
     lora_dropout=0.1,bias="none",task_type=TaskType.FEATURE_EXTRACTION
) 
    
clip_model=get_peft_model(clip_model, lora_config)

clip_model=clip_model.to(device)

##Resnet 20
target_model=torch.hub.load("chenyaofo/pytorch-cifar-models","cifar10_resnet20",pretrained=True)
target_model=target_model.to(device)
target_model.eval()

## making text vector
c10_classes=test_dataset.classes
text_prompts=[f"a photo of a {label}" for label in c10_classes]
text_inputs=clip_processor(text=[f"a photo of a {c}" for c in c10_classes], return_tensors="pt", padding=True).to(device)
text_features=clip_model.get_text_features(**text_inputs)
text_features=F.normalize(text_features, dim=-1)
text_features=text_features.detach()

Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


In [12]:
##optimizer,loss and attack
optimizer=torch.optim.AdamW(filter(lambda p: p.requires_grad, clip_model.parameters()), lr=1e-4)
tau=0.07

#Training

In [13]:
import os
save_dir = "/content/drive/MyDrive/clip_lora_TecoA_checkpoint"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
import time
import os

# Training loop
for epoch in range(15):
    clip_model.train()
    s_time=time.time()

    tr_loss=0.0
    tr_correct=0
    tr_total=0

    for imgs, labels in adv_train_loader:
        imgs, labels=imgs.to(device), labels.to(device)

        pil_imgs=[transforms.ToPILImage()(img.cpu()) for img in imgs]
        inputs=clip_processor(images=pil_imgs, return_tensors="pt")
        inputs={k: v.to(device) for k, v in inputs.items()}
        img_features=clip_model.get_image_features(**inputs)
        img_features=F.normalize(img_features, dim=-1)

        logits=(img_features @ text_features.T)/tau

        # Cross-entropy 
        loss=F.cross_entropy(logits, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tr_loss+=loss.item()
        preds=logits.argmax(dim=1)
        tr_correct+=(preds == labels).sum().item()
        tr_total+=labels.size(0)

    avg_tr_loss=tr_loss/len(adv_train_loader)
    tr_accuracy=100 * tr_correct / tr_total

    # Validation
    clip_model.eval()
    val_loss=0.0
    val_correct=0
    val_total=0


    for val_imgs, val_labels in adv_val_loader:
        val_imgs, val_labels=val_imgs.to(device), val_labels.to(device)


        with torch.no_grad():
            pil_imgs = [transforms.ToPILImage()(img.cpu()) for img in val_imgs]
            inputs=clip_processor(images=pil_imgs, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            img_features=clip_model.get_image_features(**inputs)
            img_features=F.normalize(img_features, dim=-1)

            logits=(img_features @ text_features.T)/tau
            loss=F.cross_entropy(logits, val_labels)

            val_loss+=loss.item()
            preds=logits.argmax(dim=1)
            val_correct+=(preds == val_labels).sum().item()
            val_total+=val_labels.size(0)

    avg_val_loss=val_loss/len(adv_val_loader)
    val_accuracy=100 * val_correct/val_total

    # Print epoch summary
    end_time=time.time()
    t=(end_time - s_time)/ 60
    print(f"Epoch {epoch+1} takes {t:.2f} min")
    print(f"Training Loss: {avg_tr_loss:.4f}, Training Accuracy: {tr_accuracy:.2f}%")
    print(f"Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")

    #Save model
    save_path = os.path.join(save_dir, f"clip_lora_epoch{epoch+1}.pt")
    torch.save(clip_model.state_dict(), save_path)

Epoch 1 takes 5.80 min
Training Loss: 1.7581, Training Accuracy: 71.00%
Validation Loss: 1.5773, Validation Accuracy: 82.50%
Epoch 2 takes 6.09 min
Training Loss: 1.3852, Training Accuracy: 80.20%
Validation Loss: 1.1850, Validation Accuracy: 78.50%
Epoch 3 takes 5.94 min
Training Loss: 0.9956, Training Accuracy: 84.70%
Validation Loss: 0.9059, Validation Accuracy: 84.50%
Epoch 4 takes 5.75 min
Training Loss: 0.7549, Training Accuracy: 87.90%
Validation Loss: 0.8062, Validation Accuracy: 83.00%
Epoch 5 takes 5.81 min
Training Loss: 0.6016, Training Accuracy: 91.90%
Validation Loss: 0.6538, Validation Accuracy: 87.00%
Epoch 6 takes 5.77 min
Training Loss: 0.5023, Training Accuracy: 94.00%
Validation Loss: 0.6507, Validation Accuracy: 85.50%
Epoch 7 takes 5.75 min
Training Loss: 0.4279, Training Accuracy: 95.60%
Validation Loss: 0.6646, Validation Accuracy: 84.50%
Epoch 8 takes 5.75 min
Training Loss: 0.3594, Training Accuracy: 97.40%
Validation Loss: 0.6312, Validation Accuracy: 85.00%


In [ ]:
def evaluate(loader):
    clip_model.eval()
    correct=0
    total=0

    for imgs, labels in loader:

          imgs, labels=imgs.to(device), labels.to(device)

          with torch.no_grad():

            pil_imgs=[transforms.ToPILImage()(img.cpu()) for img in imgs]

            inputs=clip_processor(images=pil_imgs, return_tensors="pt", padding=True)
            inputs={k: v.to(device) for k, v in inputs.items()}

            img_features=clip_model.get_image_features(**inputs)
            img_features=F.normalize(img_features, dim=-1)
            logits=(img_features @ text_features.T)/tau
            preds=logits.argmax(dim=1)
            correct+=(preds == labels).sum().item()
            total+=labels.size(0)

    
    accuracy_percent=100 * correct / total
    return accuracy_percent

In [ ]:
clean_acc=evaluate(test_loader2)

In [17]:
print(f"Clean Accuracy: {clean_acc:.2f}%")

Clean Accuracy: 38.50%


In [18]:
adv_acc=evaluate(adv_test_loader)

In [19]:
print(f"Adversarial Accuracy: {adv_acc:.2f}%")

Adversarial Accuracy: 87.00%
